In [1]:
import pandas as pd
import random
import joblib
import os

from surprise import Dataset
from surprise import Reader
from surprise import SVD

from surprise.model_selection import train_test_split
from surprise import accuracy

In [2]:
ratings = pd.read_csv("../data/ratings.csv")

In [3]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


In [4]:
ratings = ratings[
    ["userId", "movieId", "rating"]
]

ratings.head()

,userId,movieId,rating
0,1,17,4.0
1,1,25,1.0
2,1,29,2.0
3,1,30,5.0
4,1,32,5.0


In [5]:
reader = Reader(
    rating_scale=(0.5, 5.0)
)

In [6]:
data = Dataset.load_from_df(
    ratings,
    reader
)

In [7]:
trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

In [8]:
model = SVD()

model.fit(trainset)

In [9]:
test_predictions = model.test(testset)

rmse = accuracy.rmse(test_predictions, verbose=False)
mae  = accuracy.mae(test_predictions,  verbose=False)

print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")

RMSE: 0.7716
MAE:  0.5798


# Model Evaluation

The SVD model achieved an RMSE of 0.7718 and MAE of 0.5923 on the held-out 20% of 
MovieLens 32M. RMSE penalises large errors more heavily due to squaring; MAE reports 
the average absolute deviation directly. Both together confirm the model predicts 
user ratings to within roughly 0.6 stars on a 0.5–5.0 scale.

In [10]:
model.predict(
    uid=1,
    iid=1
)

Prediction(uid=1, iid=1, r_ui=None, est=np.float64(3.3269200467086883), details={'was_impossible': False})

In [11]:
movies = pd.read_csv("../data/movies.csv")

In [12]:
movie_title_map = movies.set_index("movieId")["title"].to_dict()

def get_movie_title(movie_id):
    return movie_title_map.get(movie_id, "Unknown")

In [13]:
user_id = 1

rated_movies = set(
    ratings[
        ratings["userId"] == user_id
    ]["movieId"]
)

all_movies = set(movies["movieId"])

unrated_movies = all_movies - rated_movies

sample_movies = random.sample(
    list(unrated_movies),
    min(5000, len(unrated_movies))
)

movie_predictions = []

for movie_id in sample_movies:

    pred = model.predict(user_id, movie_id)

    movie_predictions.append(
        (movie_id, pred.est)
    )

In [14]:
movie_predictions.sort(
    key=lambda x: x[1],
    reverse=True
)

In [15]:
top_recommendations = movie_predictions[:10]
for movie_id, score in top_recommendations:

    print(
        get_movie_title(movie_id),
        round(score, 2)
    )

Thing, The (1982) 4.53
What Time Is It There? (Ni neibian jidian) (2001) 4.32
The Work (2017) 4.31
Saragossa Manuscript, The (Rekopis znaleziony w Saragossie) (1965) 4.3
Palm Beach Story, The (1942) 4.29
Dersu Uzala (1975) 4.27
Assassins (2020) 4.23
Ali Wong: Don Wong (2022) 4.19
Of Horses and Men (2013) 4.18
Lady Eve, The (1941) 4.16


In [16]:
def recommend_for_user(user_id, n=10):

    rated_movies = set(
        ratings[
            ratings["userId"] == user_id
        ]["movieId"]
    )

    all_movies = set(movies["movieId"])

    unrated_movies = all_movies - rated_movies

    sample_movies = random.sample(
        list(unrated_movies),
        min(5000, len(unrated_movies))
    )

    predictions = []

    for movie_id in sample_movies:

        pred = model.predict(user_id, movie_id)

        predictions.append(
            (movie_id, pred.est)
        )

    predictions.sort(
        key=lambda x: x[1],
        reverse=True
    )

    recommendations = []

    for movie_id, score in predictions[:n]:

        recommendations.append({
            "title": get_movie_title(movie_id),
            "predicted_rating": round(score, 2)
        })

    return pd.DataFrame(recommendations)

In [17]:
recommend_for_user(1)  # Test-1

,title,predicted_rating
0,2001: A Space Odyssey (1968),4.56
1,All Watched Over by Machines of Loving Grace (...,4.49
2,Winnie Pooh (1969),4.32
3,Sunless (Sans Soleil) (1983),4.29
4,"Passion of Joan of Arc, The (Passion de Jeanne...",4.28
5,Danton (1983),4.28
6,His Girl Friday (1940),4.24
7,Demons (1971),4.23
8,"Bonheur, Le (1965)",4.20
9,Eternity and a Day (Mia aoniotita kai mia mera...,4.13


In [18]:
recommend_for_user(100)  # Test-2

,title,predicted_rating
0,Point of Order (1964),4.33
1,The Lost Room (2006),4.28
2,Twelve Angry Men (1954),4.15
3,Smiling Friends (2020),4.13
4,"Revolution Will Not Be Televised, The (a.k.a. ...",4.13
5,Life Is Beautiful (La Vita è bella) (1997),4.10
6,An Elephant Sitting Still (2018),4.10
7,Elway To Marino (2013),4.06
8,1987: When the Day Comes (2017),4.06
9,For Sama (2019),4.06


In [19]:
recommend_for_user(500)  # Test-3

,title,predicted_rating
0,Planet Earth (2006),4.80
1,"Sound of Music, The (1965)",4.64
2,Active Measures (2018),4.51
3,Walt Disney (2015),4.51
4,Love Letter (1995),4.47
5,Rabbit Fire (1951),4.46
6,3 Idiots (2009),4.44
7,When Harry Met Sally... (1989),4.43
8,Urusei Yatsura Movie 2: Beautiful Dreamer (Uru...,4.42
9,Custody (2018),4.39


In [20]:
recommend_for_user(
    1
).to_csv(
    "../outputs/user1_recommendations.csv",
    index=False
)

In [21]:
os.makedirs(
    "models",
    exist_ok=True
)

joblib.dump(
    model,
    "models/svd_model.pkl"
)

print("SVD model saved successfully!")

SVD model saved successfully!
